# MLflow - California Housing


In [1]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import lightgbm as lgb
from catboost import CatBoostRegressor

e:\Anaconda\envs\ML\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. Загрузка и подготовка данных

In [2]:
def eval_metrics(actual, pred):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae = mean_absolute_error(actual, pred)
    r2 = r2_score(actual, pred)
    return rmse, mae, r2

california = fetch_california_housing()
X = pd.DataFrame(california.data, columns=california.feature_names)
y = california.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

mlflow.set_experiment("California_Housing")

<Experiment: artifact_location='file:///e:/Mlops_labs/Lab3/mlruns/2', creation_time=1776082121674, experiment_id='2', last_update_time=1776082121674, lifecycle_stage='active', name='California_Housing', tags={}, trace_location=None, workspace='default'>

### 2. Эксперимент 1: LightGBM
Перебираем `learning_rate` и количество деревьев.

In [3]:
lgb_param_grid = {
    'learning_rate': [0.05, 0.1],
    'n_estimators': [100, 200],
    'max_depth': [3, 5]
}

with mlflow.start_run(run_name="LightGBM"):
    lgb_model = lgb.LGBMRegressor(random_state=42)
    lgb_grid = GridSearchCV(lgb_model, lgb_param_grid, cv=3, scoring='neg_root_mean_squared_error')
    lgb_grid.fit(X_train_scaled, y_train)
    
    best_lgb = lgb_grid.best_estimator_
    y_pred_lgb = best_lgb.predict(X_test_scaled)
    
    (rmse, mae, r2) = eval_metrics(y_test, y_pred_lgb)
    
    # Логируем ЛУЧШИЕ параметры, найденные GridSearch
    mlflow.log_params(lgb_grid.best_params_)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    
    signature = infer_signature(X_train_scaled, y_pred_lgb)
    mlflow.sklearn.log_model(best_lgb, "model", signature=signature)
    
    print(f"LightGBM Лучшие параметры: {lgb_grid.best_params_}")
    print(f"LightGBM Метрики -> RMSE: {rmse:.4f}, MAE: {mae:.4f}, R2: {r2:.4f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000205 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1837
[LightGBM] [Info] Number of data points in the train set: 11008, number of used features: 8
[LightGBM] [Info] Start training from score 2.064393
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

e:\Anaconda\envs\ML\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
e:\Anaconda\envs\ML\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
e:\Anaconda\envs\ML\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
e:\Anaconda\envs\ML\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000317 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1837
[LightGBM] [Info] Number of data points in the train set: 11008, number of used features: 8
[LightGBM] [Info] Start training from score 2.078156
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

e:\Anaconda\envs\ML\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
e:\Anaconda\envs\ML\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
e:\Anaconda\envs\ML\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

e:\Anaconda\envs\ML\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
e:\Anaconda\envs\ML\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
2026/04/13 17:21:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

2026/04/13 17:21:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


LightGBM Лучшие параметры: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}
LightGBM Метрики -> RMSE: 0.4620, MAE: 0.3074, R2: 0.8371


### 3. Эксперимент 2: CatBoost
Сравниваем с другим мощным алгоритмом градиентного бустинга.

In [4]:
cat_param_grid = {
    'learning_rate': [0.05, 0.1],
    'iterations': [100, 200],
    'depth': [4, 6]
}

with mlflow.start_run(run_name="CatBoost"):
    cat_model = CatBoostRegressor(verbose=False, random_seed=42)
    cat_grid = GridSearchCV(cat_model, cat_param_grid, cv=3, scoring='neg_root_mean_squared_error')
    cat_grid.fit(X_train_scaled, y_train)
    
    best_cat = cat_grid.best_estimator_
    y_pred_cat = best_cat.predict(X_test_scaled)
    
    (rmse, mae, r2) = eval_metrics(y_test, y_pred_cat)
    
    mlflow.log_params(cat_grid.best_params_)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    
    signature = infer_signature(X_train_scaled, y_pred_cat)
    mlflow.sklearn.log_model(best_cat, "model", signature=signature)
    
    print(f"CatBoost Лучшие параметры: {cat_grid.best_params_}")
    print(f"CatBoost Метрики -> RMSE: {rmse:.4f}, MAE: {mae:.4f}, R2: {r2:.4f}")

2026/04/13 17:21:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 17:21:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


CatBoost Лучшие параметры: {'depth': 6, 'iterations': 200, 'learning_rate': 0.1}
CatBoost Метрики -> RMSE: 0.4799, MAE: 0.3230, R2: 0.8243


### 4. Поиск лучшего запуска

In [6]:
runs = mlflow.search_runs(experiment_names=["California_Housing_Advanced"])
best_run = runs.sort_values("metrics.rmse", ascending=True).iloc[0]

print("=== ИТОГИ ===")
print(f"Лучшая модель: {best_run['tags.mlflow.runName']}")
print(f"Run ID: {best_run['run_id']}")
print(f"RMSE: {best_run['metrics.rmse']:.4f}")
print(f"R2: {best_run['metrics.r2']:.4f}")

sample = X_test_scaled[0].reshape(1, -1)
print(sample.tolist())

=== ИТОГИ ===
Лучшая модель: LightGBM
Run ID: 2bb7bb6ed5c84316a96e6564cfdca751
RMSE: 0.4620
R2: 0.8371
[[-1.155084747788599, -0.2863236926523812, -0.5206857622247829, -0.17174603248291065, -0.03030108955694076, 0.06740798113078322, 0.19509999957932808, 0.28534728163791623]]
